In [ ]:
import uproot
import matplotlib.pyplot as plt
import datetime

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT con uproot
with uproot.open(input_filename) as file:
    tree = file["press"]  # árbol correcto
    print(tree.keys())  # lista todas las columnas disponibles
    data = tree.arrays(library="np")  # Devuelve un diccionario de numpy arrays

# Extraer tiempos y valores 
times = data["t"]       # Asegúrate de que la columna exista en este árbol
press = data["press"]

print(type(times))
print(times.shape)
print(times[:5])


# Convertir tiempos a formato datetime
timestamps = [datetime.datetime.utcfromtimestamp(float(t)) for t in times]

# Graficar
plt.figure(figsize=(10, 5))
plt.plot(timestamps, press, marker='o', linestyle='-', label='Press')
plt.xlabel("Time (UTC)")
plt.ylabel("Pressure")
plt.title("Pressure vs Time")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()


In [ ]:
import uproot

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT
with uproot.open(input_filename) as file:
    print("Trees y ramas disponibles en el archivo ROOT:\n")
    
    for key in file.keys():  # recorrer todos los árboles
        # Quitar el ciclo (;n) para mostrar solo el nombre base
        tree_name = key.split(";")[0]
        print(f"Árbol: {tree_name}")
        
        tree = file[tree_name]  # abrir el árbol
        branches = tree.keys()  # obtener todas las columnas/ramas
        print(f"  Ramas: {branches}\n")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks, peak_widths

# Select polarization and sensor
pol_idx = 0   # Polarization to display
sensor_idx = 0  # Peek fiber
fiber_name = "PEEK-200 µm" if sensor_idx == 1 else "ORMOCER"

# Extract spectrum

spectrum = wav_event[pol_idx, sensor_idx, :]

# Find main peak
peaks, _ = find_peaks(spectrum)
if len(peaks) == 0:
    center_idx = len(spectrum) // 2
else:
    center_idx = peaks[np.argmax(spectrum[peaks])]

# Compute FWHM
results_half = peak_widths(spectrum, [center_idx], rel_height=0.5)
fwhm_points = results_half[0][0]
fwhm_pm = fwhm_points * 1  # assuming 1 pm per point

# Define window around the peak
window = int(fwhm_points * 2)
start_idx = max(center_idx - window, 0)
end_idx = min(center_idx + window, len(spectrum))

x_pm = np.arange(start_idx, end_idx)  # X-axis in pm
y = spectrum[start_idx:end_idx]

# Plot
plt.figure(figsize=(8,5))
plt.plot(x_pm, y, linewidth=1.5, label=f"Polarization {pol_idx} (FWHM ≈ {fwhm_pm:.1f} pm)")

# Mark FWHM
plt.hlines(results_half[1], results_half[2], results_half[3], color='red', linestyle='--')

plt.xlabel("Wavelength (pm)", fontsize=12, fontweight='bold')
plt.ylabel("Amplitude", fontsize=12, fontweight='bold')
plt.title(f"{fiber_name}", fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks, peak_widths

# wav_event shape: (n_pol, n_sens, n_pts)
n_pol, n_sens, n_pts = wav_event.shape
print(f"Polarizations: {n_pol}, Sensors: {n_sens}, Spectrum points: {n_pts}")

# ORMOCER fiber (FBGS-2)
sensor_idx = 0
fiber_name = "FBGS-2 Fiber"

# Compute mean spectrum across polarizations
mean_spectrum = wav_event[:, sensor_idx, :].mean(axis=0)

# Detect all peaks
peaks, properties = find_peaks(mean_spectrum, distance=50)  # avoid nearby tiny peaks

# Take the 4 highest peaks (main 4 sensors)
peak_heights = mean_spectrum[peaks]
top4_idx = peaks[np.argsort(peak_heights)[-4:]]  # indices of top 4 peaks
top4_idx = np.sort(top4_idx)  # sort along spectrum
print(f"Main 4 peaks at indices: {top4_idx}")

# Select which peak to zoom on (0=first, 1=second, etc.)
peak_number = 1  # change 0-3 to select different peak
center_idx = top4_idx[peak_number]

# Compute FWHM of this peak in the mean spectrum
results_half = peak_widths(mean_spectrum, [center_idx], rel_height=0.5)
fwhm_points = results_half[0][0]  # width in points
delta_lambda = 1  # 1 pm per spectrum point
fwhm_pm = fwhm_points * delta_lambda
print(f"FWHM of Peak {peak_number+1}: {fwhm_points:.1f} points ≈ {fwhm_pm:.1f} pm")

# Define window around the selected peak
window = int(fwhm_points * 2)  # ±2×FWHM for full peak
start_idx = max(center_idx - window, 0)
end_idx = min(center_idx + window, n_pts)

x = np.arange(start_idx, end_idx)

# Plot
plt.figure(figsize=(10,5))
for pol_idx in range(n_pol):
    plt.plot(x, wav_event[pol_idx, sensor_idx, start_idx:end_idx],
             linestyle='-', linewidth=1.5, label=f"Polarization {pol_idx}")

# Plot mean spectrum
plt.plot(x, mean_spectrum[start_idx:end_idx], 'k--', linewidth=2, label="Mean polarization")

# Optionally, mark FWHM
plt.hlines(results_half[1], results_half[2], results_half[3], color='red', linestyle='--', label='FWHM')

plt.xlabel("Spectrum points", fontsize=12, fontweight='bold')
plt.ylabel("Wavelength (m)", fontsize=12, fontweight='bold')
plt.title(f"{fiber_name} Spectrum\nZoomed on Peak {peak_number+1} at {event_time.strftime('%Y-%m-%d %H:%M:%S')}",
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks, peak_widths

# Select polarization and sensor
pol_idx = 0        # Polarization to display
sensor_idx = 0     # ORMOCER fiber (FBGS-2)
fiber_name = "ORMOCER"

# Extract spectrum for selected polarization
spectrum = wav_event[pol_idx, sensor_idx, :]

# Compute mean spectrum across polarizations (optional, for FWHM reference)
mean_spectrum = wav_event[:, sensor_idx, :].mean(axis=0)

# Detect all peaks in the mean spectrum
peaks, _ = find_peaks(mean_spectrum, distance=50)  # avoid nearby tiny peaks

# Take the 4 highest peaks
top4_idx = peaks[np.argsort(mean_spectrum[peaks])[-4:]]
top4_idx = np.sort(top4_idx)

# Select peak to zoom (0-3)
peak_number = 1  # second main peak
center_idx = top4_idx[peak_number]

# Compute FWHM of this peak
results_half = peak_widths(mean_spectrum, [center_idx], rel_height=0.5)
fwhm_points = results_half[0][0]
delta_lambda = 1  # 1 pm per point
fwhm_pm = fwhm_points * delta_lambda

# Define window around the peak
window = int(fwhm_points * 2)
start_idx = max(center_idx - window, 0)
end_idx = min(center_idx + window, len(spectrum))

x_pm = np.arange(start_idx, end_idx) * delta_lambda
y = spectrum[start_idx:end_idx]

# Plot
plt.figure(figsize=(8,5))
plt.plot(x_pm, y, linewidth=1.5, label=f"Polarization {pol_idx} (FWHM ≈ {fwhm_pm:.1f} pm)")

# Mark FWHM
plt.hlines(results_half[1], results_half[2]*delta_lambda, results_half[3]*delta_lambda, 
           color='red', linestyle='--')

plt.xlabel("Wavelength (pm)", fontsize=12, fontweight='bold')
plt.ylabel("Amplitude", fontsize=12, fontweight='bold')
plt.title(f"{fiber_name}", fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#creo que el espectro medio no se calcula así!!


In [ ]:
import uproot
import numpy as np
import datetime

# --- Archivo ROOT ---
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# --- Abrir el archivo ROOT ---
with uproot.open(input_filename) as file:
    peak_data = file["peak"].arrays(library="np")

wav = peak_data["wav"]  # shape: (time, polarization, sensor)
n_times, n_pols, n_sensors = wav.shape

print("=== Wavelength array structure ===")
print(f"Shape: {wav.shape}  (time, polarization, sensor)")
print(f"Number of samples: {n_times:,}")
print(f"Number of polarizations: {n_pols} (0=P, 1=S)")
print(f"Number of sensors: {n_sensors}\n")

# --- Clasificación corregida según tu descripción ---
# Fiber ORMOCER FBGs-2 → 4 FBGs (0–3)
# Fiber PEEK-3 → 1 FBG (4)
fiber_map = {
    "ORMOCER FBGs-2": [0, 1, 2, 3],
    "PEEK-3": [4]
}

print("=== Approximate wavelength centers per fiber ===")
for fiber_name, sensors in fiber_map.items():
    print(f"\nFiber: {fiber_name}")
    for s in sensors:
        mean_P = np.mean(wav[:, 0, s]) * 1e3  # µm → nm
        mean_S = np.mean(wav[:, 1, s]) * 1e3
        diff_pm = (mean_S - mean_P) * 1e3  # nm → pm
        print(f"  Sensor {s}: λ_P = {mean_P:3e} nm | λ_S = {mean_S:3e} nm | Δ(P–S) = {diff_pm:3e} pm")

# --- Mostrar los primeros valores de cada sensor ---
#print("\n=== First 5 wavelength samples per sensor ===")
#for s in range(n_sensors):
    #print(f"\nSensor {s} (P): {wav[:5, 0, s] * 1e3}")
    #print(f"Sensor {s} (S): {wav[:5, 1, s] * 1e3}")


In [ ]:
import uproot
import numpy as np
import datetime

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT
with uproot.open(input_filename) as file:
    peak_tree = file["peak"]
    press_tree = file["press"]
    temp_tree = file["temp"]
    
    peak_data = peak_tree.arrays(library="np")
    press_data = press_tree.arrays(library="np")
    temp_data = temp_tree.arrays(library="np")

# --- Tiempos de cada árbol ---
peak_times = peak_data["t"][:, 0]  # primera columna si hay varias
press_times = press_data["t"]      # normalmente 1D
temp_times = temp_data["t"]        # normalmente 1D

# --- Convertir a datetime ---
peak_start = datetime.datetime.utcfromtimestamp(peak_times[0])
press_start = datetime.datetime.utcfromtimestamp(press_times[0])
temp_start = datetime.datetime.utcfromtimestamp(temp_times[0])

print(f"Start time of Peak data: {peak_start}")
print(f"Start time of Pressure data: {press_start}")
print(f"Start time of Temperature data: {temp_start}")

# --- Primer pico de cada canal de Peak ---
n_channels_i = peak_data["wav"].shape[1]
n_channels_j = peak_data["wav"].shape[2]

for i in range(n_channels_i):
    for j in range(n_channels_j):
        wav_ = peak_data["wav"][:, i, j]
        first_idx = np.argmax(wav_ > 0)  # primer valor mayor que cero
        first_time = datetime.datetime.utcfromtimestamp(peak_times[first_idx])
        print(f"Channel [{i},{j}] first peak timestamp: {first_time}")


In [ ]:
# --- Convertir a datetime (con corrección de desfase de 2 horas en peak) ---
time_shift = datetime.timedelta(hours=2)

peak_start = datetime.datetime.utcfromtimestamp(peak_times[0]) + time_shift
press_start = datetime.datetime.utcfromtimestamp(press_times[0])
temp_start = datetime.datetime.utcfromtimestamp(temp_times[0])

print(f"Start time of Peak data: {peak_start}")
print(f"Start time of Pressure data: {press_start}")
print(f"Start time of Temperature data: {temp_start}")

# --- Primer pico de cada canal de Peak ---
for i in range(n_channels_i):
    for j in range(n_channels_j):
        wav_ = peak_data["wav"][:, i, j]
        first_idx = np.argmax(wav_ > 0)  # primer valor mayor que cero
        first_time = datetime.datetime.utcfromtimestamp(peak_times[first_idx]) + time_shift
        print(f"Channel [{i},{j}] first peak timestamp: {first_time}")


In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT
with uproot.open(input_filename) as file:
    tree = file["peak"]
    data = tree.arrays(library="np")

# Extraer tiempos (primera columna si hay varias) y sumar 2 horas para corregir desfase
times = data["t"][:, 0]
timestamps = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2) for t in times])

# Función para graficar un canal de wav automáticamente
def plot_wav_channel(i, j, label, umbral_min=None, umbral_max=None):
    wav_ = data["wav"][:, i, j]
    
    # Si no se pasan umbrales, usar 5% por debajo y por encima de los valores
    if umbral_min is None:
        umbral_min = np.min(wav_) - 0.05 * np.ptp(wav_)
    if umbral_max is None:
        umbral_max = np.max(wav_) + 0.05 * np.ptp(wav_)
    
    mask = (wav_ > umbral_min) & (wav_ < umbral_max)
    print(f"Channel {label}: {np.sum(mask)} points plotted (min={np.min(wav_):.6e}, max={np.max(wav_):.6e})")
    
    plt.figure(figsize=(10,5))
    plt.plot(timestamps[mask], wav_[mask], marker='o', linestyle='-', label=label)
    plt.xlabel("Time (UTC)")
    plt.ylabel("Wavelength (m)")
    plt.title(f"Peaks - Channel {label}")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid()
    plt.show()

# Graficar los dos primeros canales
plot_wav_channel(0, 4, "wav_04")
plot_wav_channel(1, 3, "wav_13")


In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT
with uproot.open(input_filename) as file:
    tree = file["peak"]
    data = tree.arrays(library="np")

# Extraer tiempos (primera columna si hay varias) y corregir desfase de 2 horas
times = data["t"][:, 0]
timestamps = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2) for t in times])

# Función para graficar un canal de wav automáticamente con filtrado por intervalo de tiempo
def plot_wav_channel(i, j, label, umbral_min=None, umbral_max=None, start_time=None, end_time=None):
    wav_ = data["wav"][:, i, j]
    
    # Si no se pasan umbrales, usar 5% por debajo y por encima de los valores
    if umbral_min is None:
        umbral_min = np.min(wav_) - 0.05 * np.ptp(wav_)
    if umbral_max is None:
        umbral_max = np.max(wav_) + 0.05 * np.ptp(wav_)
    
    # Filtrar por umbral
    mask = (wav_ > umbral_min) & (wav_ < umbral_max)
    
    # Filtrar por intervalo temporal
    if start_time is not None:
        mask &= (timestamps >= start_time)
    if end_time is not None:
        mask &= (timestamps <= end_time)
    
    print(f"Channel {label}: {np.sum(mask)} points plotted (min={np.min(wav_):.6e}, max={np.max(wav_):.6e})")
    
    plt.figure(figsize=(10,5))
    plt.plot(timestamps[mask], wav_[mask], marker='o', linestyle='-', label=label)
    plt.xlabel("Time (UTC)")
    plt.ylabel("Wavelength (m)")
    plt.title(f"Peaks - Channel {label}")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid()
    plt.show()

# Definir hora de inicio del plot
start_time = datetime.datetime.strptime("2025-05-15 12:30:00", "%Y-%m-%d %H:%M:%S")

# Graficar los canales deseados
plot_wav_channel(0, 0, "wav_00 ormocer fbgs-2", start_time=start_time)
plot_wav_channel(1, 4, "wav_14 peek-3-alt", start_time=start_time)
plot_wav_channel(0, 4, "wav_04 peek-3-alt", start_time=start_time)


In [ ]:
# -------------------------
# PLOT DE DOS POLARIZACIONES + MEDIA
# -------------------------
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT
with uproot.open(input_filename) as file:
    tree = file["peak"]
    data = tree.arrays(library="np")

# Extraer tiempos (primera columna si hay varias) y corregir desfase de 2 horas
times = data["t"][:, 0]
timestamps = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2) for t in times])

# Seleccionar los canales de interés
wav_04 = data["wav"][:, 0, 4]  # polarización 0
wav_14 = data["wav"][:, 1, 4]  # polarización 1

# Definir hora de inicio del plot
start_time = datetime.datetime.strptime("2025-05-15 12:20:00", "%Y-%m-%d %H:%M:%S")
mask_time = timestamps >= start_time

# Filtramos solo los puntos desde start_time
timestamps_plot = timestamps[mask_time]
wav_04_plot = wav_04[mask_time]
wav_14_plot = wav_14[mask_time]

# Calculamos la media de ambas polarizaciones
wav_mean_plot = (wav_04_plot + wav_14_plot) / 2

# -------------------------
# Plot combinado
# -------------------------
plt.figure(figsize=(12,6))

plt.plot(timestamps_plot, wav_04_plot, marker='o', linestyle='-', label='Pol. s')
plt.plot(timestamps_plot, wav_14_plot, marker='s', linestyle='-', label='Pol. p')
plt.plot(timestamps_plot, wav_mean_plot, marker='^', linestyle='--', color='black', label='Avg Pol.')

plt.xlabel("Tiempo (UTC)")
plt.ylabel("Longitud de onda (m)")
plt.title("Sensor of fiber PEEK-ALT")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Graficar wav[0][1]
plot_wav_channel(0, 1, "wav_01", start_time=start_time)

# Graficar wav[1][1]
plot_wav_channel(1, 1, "wav_11", start_time=start_time)


In [ ]:
# Graficar wav[0][2] automáticamente ajustando el umbral
plot_wav_channel(0, 2, "wav_02", start_time=start_time)

# Graficar wav[1][2] automáticamente ajustando el umbral
plot_wav_channel(1, 2, "wav_12", start_time=start_time)

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Archivo ROOT de entrada
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"

# Abrir el archivo ROOT y cargar el árbol "temp"
with uproot.open(input_filename) as file:
    tree = file["temp"]
    data = tree.arrays(library="np")

# Extraer temperaturas
temp = data["temp"]

# Convertir tiempos a formato datetime
timestamps = np.array([datetime.datetime.utcfromtimestamp(t) for t in data["t"]])  # ya es 1D

# 🔹 Graficar las temperaturas de los primeros 4 sensores (fuera de la cápsula)
plt.figure(figsize=(10, 5))
for i in range(4):
    plt.plot(timestamps, temp[:, i], marker='o', linestyle='-', label=f'Temp Sensor {i+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title("First 4 RTD Sensors Temperature (Level Meter)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()

# 🔹 Graficar un sensor específico (ej. último sensor)
sensor_id = 7  # índice del último sensor
plt.figure(figsize=(10, 5))
plt.plot(timestamps, temp[:, sensor_id], marker='o', linestyle='-', color='r', label=f'Temp Sensor {sensor_id+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title(f"RTD Sensor {sensor_id+1} Temperature (INSIDE THE CAPSULE)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()

# 🔹 Graficar un sensor específico (ej. penúltimo sensor)
sensor_id = 6  # índice del penúltimo sensor
plt.figure(figsize=(10, 5))
plt.plot(timestamps, temp[:, sensor_id], marker='o', linestyle='-', color='r', label=f'Temp Sensor {sensor_id+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title(f"RTD Sensor {sensor_id+1} Temperature (INSIDE THE CAPSULE)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()


In [ ]:
# --- Definir hora de inicio del plot ---
start_time = datetime.datetime.strptime("2025-05-15 12:40:00", "%Y-%m-%d %H:%M:%S")

# --- Crear máscara temporal ---
mask_time = timestamps >= start_time

# 🔹 Graficar el último sensor (ej. sensor 8, índice 7)
sensor_id = 7
plt.figure(figsize=(10, 5))
plt.plot(timestamps[mask_time], temp[:, sensor_id][mask_time],
         marker='o', linestyle='-', color='r', label=f'Temp Sensor {sensor_id+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title(f"RTD Sensor {sensor_id+1} Temperature (INSIDE THE CAPSULE)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()

# 🔹 Graficar el penúltimo sensor (ej. sensor 7, índice 6)
sensor_id = 6
plt.figure(figsize=(10, 5))
plt.plot(timestamps[mask_time], temp[:, sensor_id][mask_time],
         marker='o', linestyle='-', color='r', label=f'Temp Sensor {sensor_id+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title(f"RTD Sensor {sensor_id+1} Temperature (INSIDE THE CAPSULE)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()


In [ ]:
# 🔹 Graficar un sensor específico (ej. antepenúltimo sensor)
sensor_id = 5  # índice del antepenúltimo sensor
plt.figure(figsize=(10, 5))
plt.plot(timestamps, temp[:, sensor_id], marker='o', linestyle='-', color='r', label=f'Temp Sensor {sensor_id+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title(f"RTD Sensor {sensor_id+1} Temperature (EMPTY)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()

# 🔹 Graficar un sensor específico (ej.)
sensor_id = 4  # índice 
plt.figure(figsize=(10, 5))
plt.plot(timestamps, temp[:, sensor_id], marker='o', linestyle='-', color='r', label=f'Temp Sensor {sensor_id+1}')
plt.xlabel("Time (UTC)")
plt.ylabel("Temperature (K)")
plt.title(f"RTD Sensor {sensor_id+1} Temperature (EMPTY)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime

def find_plateaus(values, times, tolerance=0.01, min_plateau_length=500):
    """
    Encuentra plateaus en los datos.
    - values: array de valores.
    - times: array de tiempos en segundos.
    - tolerance: diferencia máxima para considerar un plateau.
    - min_plateau_length: duración mínima del plateau en segundos.
    """
    plateaus = []
    start_idx = 0
    for i in range(1, len(values)):
        if abs(values[i] - values[start_idx]) > tolerance:
            duration = times[i - 1] - times[start_idx]
            if duration >= min_plateau_length:
                segment = values[start_idx:i]
                plateaus.append({
                    "t0": times[start_idx],
                    "tfin": times[i - 1],
                    "mean": np.mean(segment),
                    "std": np.std(segment),
                    "data": segment
                })
            start_idx = i
    return plateaus

def calculate_plateaus(input_filename, sensor_temp=0, sensor_wav=(0,0),
                       params=None, start_time=None):
    """
    Calcula plateaus para press, temp y wav a partir de un archivo ROOT.
    Permite filtrar los datos por un tiempo mínimo 'start_time'.
    """
    if params is None:
        params = {
            "press": {"tolerance": 0.01, "min_plateau_length": 500},
            "temp": {"tolerance": 0.01, "min_plateau_length": 500},
            "wav": {"tolerance": 0.01, "min_plateau_length": 500}
        }
    
    with uproot.open(input_filename) as file:
        peak_tree = file["peak"]
        press_tree = file["press"]
        temp_tree = file["temp"]
        
        peak_data = peak_tree.arrays(library="np")
        press_data = press_tree.arrays(library="np")
        temp_data = temp_tree.arrays(library="np")
    
    # --- Tiempos corregidos ---
    peak_times  = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2)
                            for t in peak_data["t"][:, 0]])  # primera polarización
    press_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in press_data["t"]])
    temp_times  = np.array([datetime.datetime.utcfromtimestamp(t) for t in temp_data["t"]])
    
    # --- Filtrado temporal si se define start_time ---
    if start_time is not None:
        mask_peak  = peak_times >= start_time
        mask_press = press_times >= start_time
        mask_temp  = temp_times >= start_time
    else:
        mask_peak  = np.ones_like(peak_times, dtype=bool)
        mask_press = np.ones_like(press_times, dtype=bool)
        mask_temp  = np.ones_like(temp_times, dtype=bool)
    
    # --- Datos filtrados ---
    peak_times  = np.array([t.timestamp() for t in peak_times[mask_peak]])
    press_times = np.array([t.timestamp() for t in press_times[mask_press]])
    temp_times  = np.array([t.timestamp() for t in temp_times[mask_temp]])
    
    peak_values = peak_data["wav"][mask_peak, sensor_wav[0], sensor_wav[1]] * 1e6  # micro m
    press_values = press_data["press"][mask_press]
    temp_values  = temp_data["temp"][mask_temp, sensor_temp]
    
    # --- Buscar plateaus ---
    press_plateaus = find_plateaus(press_values, press_times, **params["press"])
    temp_plateaus  = find_plateaus(temp_values, temp_times, **params["temp"])
    wav_plateaus   = find_plateaus(peak_values, peak_times, **params["wav"])
    
    # --- Graficar ---
    for branch, times_arr, values_arr, plateaus, color in zip(
        ["press", "temp", "wav"],
        [press_times, temp_times, peak_times],
        [press_values, temp_values, peak_values],
        [press_plateaus, temp_plateaus, wav_plateaus],
        ["blue", "green", "orange"]):
        
        plt.figure(figsize=(10,5))
        plt.plot([datetime.datetime.fromtimestamp(t) for t in times_arr], values_arr, marker='o', linestyle='-', label=branch, color=color)
        for p in plateaus:
            plt.axvline(datetime.datetime.fromtimestamp(p["t0"]), color='red', linestyle='--', alpha=0.5)
            plt.axvline(datetime.datetime.fromtimestamp(p["tfin"]), color='blue', linestyle='--', alpha=0.5)
        plt.xlabel("Time (UTC)")
        plt.ylabel(branch)
        plt.title(f"{branch} vs Time with Plateaus")
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid()
        plt.show()
    
    # --- Plateaus comunes (intersección de intervalos) ---
    common_plateaus = []
    for p_press in press_plateaus:
        for p_temp in temp_plateaus:
            for p_wav in wav_plateaus:
                t_start = max(p_press["t0"], p_temp["t0"], p_wav["t0"])
                t_end   = min(p_press["tfin"], p_temp["tfin"], p_wav["tfin"])
                if t_end > t_start:
                    common_plateaus.append({"t0": t_start, "tfin": t_end})
    
    print("\nPlateaus comunes entre press, temp y wav:")
    for i, p in enumerate(common_plateaus):
        print(f"{i}: {datetime.datetime.fromtimestamp(p['t0'])} → {datetime.datetime.fromtimestamp(p['tfin'])}")
    
    return {"press": press_plateaus, "temp": temp_plateaus, "wav": wav_plateaus, "common": common_plateaus}

# --- Ejemplo de uso ---
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"
start_time = datetime.datetime.strptime("2025-05-15 12:30:00", "%Y-%m-%d %H:%M:%S")

params = {
    "press": {"tolerance": 0.25, "min_plateau_length": 1000},
    "temp": {"tolerance": 0.5, "min_plateau_length": 800},
    "wav": {"tolerance": 5e-6, "min_plateau_length": 800}
}

plateaus_result = calculate_plateaus(input_filename, sensor_temp=7, sensor_wav=(1,0),
                                     params=params, start_time=start_time)


In [ ]:
# Mostrar los plateaus comunes detectados
for i, p in enumerate(plateaus_result["common"]):
    print(f"{i}: {datetime.datetime.fromtimestamp(p['t0'])} → {datetime.datetime.fromtimestamp(p['tfin'])}")

# Añadir manualmente 2 plateaus extra
manual_plateaus = [
    {"t0": datetime.datetime(2025,5,15,14,40,0).timestamp(), "tfin": datetime.datetime(2025,5,15,14,45,0).timestamp()},
    {"t0": datetime.datetime(2025,5,15,14,50,0).timestamp(), "tfin": datetime.datetime(2025,5,15,14,54,0).timestamp()}
]

plateaus_result["common"].extend(manual_plateaus)

# Ordenar por tiempo de inicio
plateaus_result["common"] = sorted(plateaus_result["common"], key=lambda x: x["t0"])

# Verificar
print("\nLista final de plateaus (detectados + manuales):")
for i, p in enumerate(plateaus_result["common"]):
    print(f"{i}: {datetime.datetime.fromtimestamp(p['t0'])} → {datetime.datetime.fromtimestamp(p['tfin'])}")


In [ ]:
# Lista de sensores FBG a analizar (tupla de índices: (i,j))
fbgs_to_check = [(0,0), (0,1), (1,1), (1,4)]  # añadir los que quieras

# Recorrer cada FBG y mostrar los plateaus comunes
for sensor_wav in fbgs_to_check:
    print(f"\n=== Sensor FBG {sensor_wav} ===")
    
    plateaus_result = calculate_plateaus(
        input_filename,
        sensor_temp=7,          # mismo sensor RTD que antes
        sensor_wav=sensor_wav,
        params=params,
        start_time=start_time
    )
    
    common = plateaus_result["common"]
    if len(common) == 0:
        print("No common plateaus found for this FBG.")
    else:
        print("Common plateaus (press, temp, wav):")
        for i, p in enumerate(common):
            t0 = datetime.datetime.fromtimestamp(p["t0"])
            tfin = datetime.datetime.fromtimestamp(p["tfin"])
            print(f"{i}: {t0} → {tfin}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import datetime
import uproot

# --- Parameters ---
sensor_temp = 7  # RTD sensor, se puede cambiar al 6 tambien
fbgs_to_check = [(0,0), (0,1), (1,0), (1,1)]  # Example list of FBGs and polarizations

# Open ROOT file and get data
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"
with uproot.open(input_filename) as file:
    peak_data = file["peak"].arrays(library="np")
    temp_data = file["temp"].arrays(library="np")

# --- Convert times to datetime ---
temp_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in temp_data["t"]])
peak_times = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2) for t in peak_data["t"][:,0]])

# --- Iterate over FBG sensors ---
for sensor_wav in fbgs_to_check:
    temp_means = []
    temp_stds  = []
    wav_means  = []
    wav_stds   = []

    # --- Iterate over common plateaus ---
    common_plateaus = plateaus_result["common"]

    for p in common_plateaus:
        t0, tfin = datetime.datetime.fromtimestamp(p["t0"]), datetime.datetime.fromtimestamp(p["tfin"])
        
        # Filter indices of temp and wav within the plateau
        temp_idx = np.where((temp_times >= t0) & (temp_times <= tfin))[0]
        wav_idx  = np.where((peak_times >= t0) & (peak_times <= tfin))[0]
        
        if len(temp_idx) == 0 or len(wav_idx) == 0:
            continue  # Skip if no data in plateau
        
        # Compute means and standard deviations
        temp_vals = temp_data["temp"][temp_idx, sensor_temp]
        wav_vals  = peak_data["wav"][wav_idx, sensor_wav[0], sensor_wav[1]] * 1e6  # µm
        
        temp_means.append(np.mean(temp_vals))
        temp_stds.append(np.std(temp_vals))
        wav_means.append(np.mean(wav_vals))
        wav_stds.append(np.std(wav_vals))

    if len(temp_means) == 0:
        print(f"No data found for FBG {sensor_wav}")
        continue

    # --- Linear fit ---
    slope, intercept, r_value, p_value, std_err = linregress(temp_means, wav_means)
    fit_line = [slope * x + intercept for x in temp_means]

    # --- Plot ---
    plt.figure(figsize=(10, 7))
    plt.errorbar(temp_means, wav_means, xerr=temp_stds, yerr=wav_stds,
                 fmt='o', color='royalblue', ecolor='gray', capsize=4, markersize=6,
                 label="Plateau averages")
    plt.plot(temp_means, fit_line, color='firebrick', linestyle='--',
             label=f'Linear fit: slope = {slope:.4e} µm/K ± {std_err:.4e}')

    # Labels and title
    fbg_label = f"FBG {sensor_wav[0]+1} Pol. {sensor_wav[1]+1}"
    plt.xlabel("Mean Plateau Temperature (K)", fontsize=14, fontweight='bold')
    plt.ylabel("Mean Plateau Wavelength (µm)", fontsize=14, fontweight='bold')
    plt.title(f"Linear Fit {fbg_label} vs RTD {sensor_temp+1}", fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"{fbg_label} linear fit slope: {slope:.4e} µm/K ± {std_err:.4e}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import datetime
import uproot

def analyze_fiber_plateaus(input_filename, plateaus_result, sensor_temp=7):
    """
    Performs linear fits for each fiber, sensor, and polarization
    using plateau-averaged wavelength and temperature data.
    """

    # --- Open ROOT file and load data ---
    with uproot.open(input_filename) as file:
        peak_data = file["peak"].arrays(library="np")
        temp_data = file["temp"].arrays(library="np")

    # --- Convert timestamps ---
    temp_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in temp_data["t"]])
    peak_times = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2)
                           for t in peak_data["t"][:, 0]])

    # --- Fiber and sensor mapping ---
    fiber_map = {
        "ORMOCER FBGs-2": [0, 1, 2, 3],
        "PEEK-3": [4]
    }

    polarizations = {0: "P", 1: "S"}

    # --- Loop through fibers, sensors, and polarizations ---
    for fiber_name, sensors in fiber_map.items():
        print(f"\n=== Fiber: {fiber_name} ===")

        for s in sensors:
            for pol in polarizations.keys():
                temp_means, temp_stds = [], []
                wav_means, wav_stds = [], []

                for p in plateaus_result["common"]:
                    t0 = datetime.datetime.fromtimestamp(p["t0"])
                    tfin = datetime.datetime.fromtimestamp(p["tfin"])

                    # Filter indices inside plateau
                    temp_idx = np.where((temp_times >= t0) & (temp_times <= tfin))[0]
                    wav_idx = np.where((peak_times >= t0) & (peak_times <= tfin))[0]

                    if len(temp_idx) == 0 or len(wav_idx) == 0:
                        continue

                    # Extract values
                    temp_vals = temp_data["temp"][temp_idx, sensor_temp]
                    wav_vals = peak_data["wav"][wav_idx, pol, s] * 1e6  # µm

                    # Store means and stds
                    temp_means.append(np.mean(temp_vals))
                    temp_stds.append(np.std(temp_vals))
                    wav_means.append(np.mean(wav_vals))
                    wav_stds.append(np.std(wav_vals))

                # --- Skip empty results ---
                if len(temp_means) == 0:
                    print(f"   Sensor {s} ({polarizations[pol]}) → No data found")
                    continue

                # --- Linear fit ---
                slope, intercept, r_value, p_value, std_err = linregress(temp_means, wav_means)
                fit_line = [slope * x + intercept for x in temp_means]

                # --- Plot ---
                plt.figure(figsize=(9, 6))
                plt.errorbar(temp_means, wav_means, xerr=temp_stds, yerr=wav_stds,
                             fmt='o', color='royalblue', ecolor='gray', capsize=4, markersize=6,
                             label="Plateau averages")
                plt.plot(temp_means, fit_line, color='firebrick', linestyle='--',
                         label=f'Linear fit: slope = {slope:.3e} ± {std_err:.1e} µm/K')

                plt.xlabel("Mean Plateau Temperature (K)", fontsize=13, fontweight='bold')
                plt.ylabel("Mean Plateau Wavelength (µm)", fontsize=13, fontweight='bold')
                plt.title(f"{fiber_name} - FBG {s} ({polarizations[pol]} Pol.) vs RTD {sensor_temp+1}",
                          fontsize=15, fontweight='bold')
                plt.legend(fontsize=11)
                plt.grid(alpha=0.3)
                plt.tight_layout()
                plt.show()

                print(f"   Sensor {s} ({polarizations[pol]}) slope: {slope:.4e} µm/K ± {std_err:.4e}")

# --- Example of usage ---
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"
analyze_fiber_plateaus(input_filename, plateaus_result, sensor_temp=7)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import datetime
import uproot

# --- Parameters ---
sensor_temp = 6  # RTD sensor, se puede cambiar al 7 tambien
fbgs_to_check = [(0,0), (0,1), (1,0), (1,1)]  # Example list of FBGs and polarizations

# Open ROOT file and get data
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"
with uproot.open(input_filename) as file:
    peak_data = file["peak"].arrays(library="np")
    temp_data = file["temp"].arrays(library="np")

# --- Convert times to datetime ---
temp_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in temp_data["t"]])
peak_times = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2) for t in peak_data["t"][:,0]])

# --- Iterate over FBG sensors ---
for sensor_wav in fbgs_to_check:
    temp_means = []
    temp_stds  = []
    wav_means  = []
    wav_stds   = []

    # --- Iterate over common plateaus ---
    common_plateaus = plateaus_result["common"]

    for p in common_plateaus:
        t0, tfin = datetime.datetime.fromtimestamp(p["t0"]), datetime.datetime.fromtimestamp(p["tfin"])
        
        # Filter indices of temp and wav within the plateau
        temp_idx = np.where((temp_times >= t0) & (temp_times <= tfin))[0]
        wav_idx  = np.where((peak_times >= t0) & (peak_times <= tfin))[0]
        
        if len(temp_idx) == 0 or len(wav_idx) == 0:
            continue  # Skip if no data in plateau
        
        # Compute means and standard deviations
        temp_vals = temp_data["temp"][temp_idx, sensor_temp]
        wav_vals  = peak_data["wav"][wav_idx, sensor_wav[0], sensor_wav[1]] * 1e6  # µm
        
        temp_means.append(np.mean(temp_vals))
        temp_stds.append(np.std(temp_vals))
        wav_means.append(np.mean(wav_vals))
        wav_stds.append(np.std(wav_vals))

    if len(temp_means) == 0:
        print(f"No data found for FBG {sensor_wav}")
        continue

    # --- Linear fit ---
    slope, intercept, r_value, p_value, std_err = linregress(temp_means, wav_means)
    fit_line = [slope * x + intercept for x in temp_means]

    # --- Plot ---
    plt.figure(figsize=(10, 7))
    plt.errorbar(temp_means, wav_means, xerr=temp_stds, yerr=wav_stds,
                 fmt='o', color='royalblue', ecolor='gray', capsize=4, markersize=6,
                 label="Plateau averages")
    plt.plot(temp_means, fit_line, color='firebrick', linestyle='--',
             label=f'Linear fit: slope = {slope:.4e} µm/K ± {std_err:.4e}')

    # Labels and title
    fbg_label = f"FBG {sensor_wav[0]+1} Pol. {sensor_wav[1]+1}"
    plt.xlabel("Mean Plateau Temperature (K)", fontsize=14, fontweight='bold')
    plt.ylabel("Mean Plateau Wavelength (µm)", fontsize=14, fontweight='bold')
    plt.title(f"Linear Fit {fbg_label} vs RTD {sensor_temp+1}", fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"{fbg_label} linear fit slope: {slope:.4e} µm/K ± {std_err:.4e}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import datetime
import uproot

# --- Choose sensors ---
sensor_temp = 7
ormocer_sensor = 1  # sensor from ORMOCER FBGs-2
peek_sensor = 4     # sensor from PEEK-3

# --- Open ROOT file and load data ---
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"
with uproot.open(input_filename) as file:
    peak_data = file["peak"].arrays(library="np")
    temp_data = file["temp"].arrays(library="np")

# --- Convert times ---
temp_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in temp_data["t"]])
peak_times = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2)
                       for t in peak_data["t"][:, 0]])

# --- Prepare data for two sensors ---
def extract_plateau_means(sensor_wav, plateaus_result):
    temp_means, temp_stds = [], []
    wav_means, wav_stds = [], []

    for p in plateaus_result["common"]:
        t0, tfin = datetime.datetime.fromtimestamp(p["t0"]), datetime.datetime.fromtimestamp(p["tfin"])
        
        temp_idx = np.where((temp_times >= t0) & (temp_times <= tfin))[0]
        wav_idx  = np.where((peak_times >= t0) & (peak_times <= tfin))[0]

        if len(temp_idx) == 0 or len(wav_idx) == 0:
            continue

        temp_vals = temp_data["temp"][temp_idx, sensor_temp]
        wav_vals = peak_data["wav"][wav_idx, 0, sensor_wav] * 1e6  # µm, using Pol P (0)

        temp_means.append(np.mean(temp_vals))
        temp_stds.append(np.std(temp_vals))
        wav_means.append(np.mean(wav_vals))
        wav_stds.append(np.std(wav_vals))

    return np.array(temp_means), np.array(temp_stds), np.array(wav_means), np.array(wav_stds)

# --- Extract plateau means ---
t_mean_orm, t_std_orm, wav_mean_orm, wav_std_orm = extract_plateau_means(ormocer_sensor, plateaus_result)
t_mean_peek, t_std_peek, wav_mean_peek, wav_std_peek = extract_plateau_means(peek_sensor, plateaus_result)

# --- Linear fits ---
slope_orm, intercept_orm, _, _, std_err_orm = linregress(t_mean_orm, wav_mean_orm)
slope_peek, intercept_peek, _, _, std_err_peek = linregress(t_mean_peek, wav_mean_peek)

fit_orm = slope_orm * t_mean_orm + intercept_orm
fit_peek = slope_peek * t_mean_peek + intercept_peek

# --- Plot ---
plt.figure(figsize=(10, 7))
plt.errorbar(t_mean_orm, wav_mean_orm, xerr=t_std_orm, yerr=wav_std_orm, fmt='o',
             color='royalblue', ecolor='gray', capsize=4, markersize=6, label='ORMOCER FBGs-2 Sensor 1')
plt.plot(t_mean_orm, fit_orm, 'b--', label=f'ORMOCER fit: slope={slope_orm:.4e} µm/K')

plt.errorbar(t_mean_peek, wav_mean_peek, xerr=t_std_peek, yerr=wav_std_peek, fmt='o',
             color='darkorange', ecolor='gray', capsize=4, markersize=6, label='PEEK-3 Sensor 4')
plt.plot(t_mean_peek, fit_peek, 'r--', label=f'PEEK fit: slope={slope_peek:.4e} µm/K')

plt.xlabel("Mean Plateau Temperature (K)", fontsize=13, fontweight='bold')
plt.ylabel("Mean Plateau Wavelength (µm)", fontsize=13, fontweight='bold')
plt.title("Comparison of Linear Fit: ORMOCER vs PEEK-3", fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ORMOCER Sensor 1 slope: {slope_orm:.4e} µm/K ± {std_err_orm:.4e}")
print(f"PEEK-3 Sensor 4 slope: {slope_peek:.4e} µm/K ± {std_err_peek:.4e}")
